# 05 — Геометрический анализ: расчёт потери кости

Проверка правильности работы `stage2_measure.py` на аннотированных примерах.

Ключевые величины:
- **CEJ** (cemento-enamel junction) — граница корня и коронки: ~40% от верхушки зуба
- **bone_crest_y** — верхняя граница альвеолярной кости в столбцах зуба
- **bone_loss_ratio** = (bone_crest_y − CEJ_y) / root_length

In [ ]:
import sys; sys.path.insert(0, '..')
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.pipeline.stage2_measure import measure_bone_loss
from src.utils.visualization import overlay_masks

IMG_DIR  = Path('../data/processed/images')
MASK_DIR = Path('../data/processed/masks')

CLASS_TOOTH    = 1
CLASS_BONE     = 2
CLASS_CORTICAL = 3

In [ ]:
def load_sample(image_id: str):
    img  = cv2.cvtColor(cv2.imread(str(IMG_DIR / f'{image_id}.png')), cv2.COLOR_BGR2RGB)
    mask = cv2.imread(str(MASK_DIR / f'{image_id}.png'), cv2.IMREAD_GRAYSCALE)
    return img, mask

sample_ids = [p.stem for p in sorted(IMG_DIR.glob('*.png'))[:5]]

for sid in sample_ids:
    img, mask = load_sample(sid)
    m_tooth    = (mask == CLASS_TOOTH)
    m_bone     = (mask == CLASS_BONE)
    m_cortical = (mask == CLASS_CORTICAL)

    measurements = measure_bone_loss(m_tooth, m_bone, m_cortical)

    vis = overlay_masks(img, mask, measurements)
    plt.figure(figsize=(14, 4))
    plt.imshow(vis)
    plt.title(f'{sid} — {len(measurements)} teeth detected')
    plt.axis('off')
    plt.show()

    for m in measurements:
        print(f'  Tooth {m.tooth_id}: loss={m.bone_loss_ratio:.1%}, cortical={m.cortical_coverage:.1%}')

In [ ]:
# Validate against manual ground truth
# Fill in: {image_id: {tooth_id: expected_bone_loss_ratio}}
GROUND_TRUTH = {
    # 'sample_001': {1: 0.25, 2: 0.50},
}

if GROUND_TRUTH:
    errors = []
    for sid, gt in GROUND_TRUTH.items():
        img, mask = load_sample(sid)
        measurements = measure_bone_loss(
            mask == CLASS_TOOTH, mask == CLASS_BONE, mask == CLASS_CORTICAL
        )
        for m in measurements:
            if m.tooth_id in gt:
                errors.append(abs(m.bone_loss_ratio - gt[m.tooth_id]))
    print(f'MAE on bone_loss_ratio vs manual GT: {np.mean(errors):.3f}')
else:
    print('Add ground truth above to run validation.')